# SARSA versus Q-learning: the next-action term

Companion to HTB Academy Module 290, Section 15. SARSA means **State–Action–Reward–State–Action** (say “SAR-suh”). Read s-prime as **the next state** and a-prime as **the actual next action selected by the current policy**. Alpha is the learning rate; gamma discounts future value. This notebook compares one supplied transition, not an RL training run.

## Hold the observed transition constant

Both methods start at S1, take Right, receive reward 0.5, and arrive at S2. In S2, Down has the largest Q estimate (1.0), but suppose the exploratory behavior policy actually chooses Right (−0.3). SARSA uses the **chosen** next action; Q-learning uses the **largest** next-state action value. Both target calculations use the same unmodified table snapshot.

In [ ]:
q = {
    'S1': {'Up': -1.0, 'Down': 0.0, 'Left': -0.5, 'Right': 0.2},
    'S2': {'Up': 0.0, 'Down': 1.0, 'Left': 0.0, 'Right': -0.3},
}
state, action = 'S1', 'Right'
next_state, chosen_next_action = 'S2', 'Right'  # Actual exploratory choice a-prime.
reward, alpha, gamma = 0.5, 0.1, 0.9
old_q = q[state][action]
next_chosen_q = q[next_state][chosen_next_action]
next_best_q = max(q[next_state].values())

# SARSA target = r + gamma * Q(s-prime, ACTUALLY CHOSEN a-prime).
sarsa_target = reward + gamma * next_chosen_q
sarsa_error = sarsa_target - old_q
sarsa_new = old_q + alpha * sarsa_error
# Q-learning target = r + gamma * max over possible next actions.
q_learning_target = reward + gamma * next_best_q
q_learning_error = q_learning_target - old_q
q_learning_new = old_q + alpha * q_learning_error
print('next chosen:', next_chosen_q, 'next best:', next_best_q)
print('SARSA target/error/new:', sarsa_target, sarsa_error, sarsa_new)
print('Q-learning target/error/new:', q_learning_target, q_learning_error, q_learning_new)
assert abs(sarsa_target - 0.23) < 1e-12
assert abs(sarsa_new - 0.203) < 1e-12
assert abs(q_learning_target - 1.4) < 1e-12
assert abs(q_learning_new - 0.32) < 1e-12
assert sarsa_new < q_learning_new

## When the actual next action is greedy

If the policy really chooses S2/Down, SARSA and Q-learning have the same one-step target here. Their difference appears when the action **actually selected** differs from the highest-Q action. On-policy does not itself prevent unsafe exploration.

In [ ]:
greedy_next_action = 'Down'
greedy_sarsa_target = reward + gamma * q[next_state][greedy_next_action]
greedy_sarsa_new = old_q + alpha * (greedy_sarsa_target - old_q)
print('SARSA with greedy actual next action:', greedy_sarsa_new)
assert greedy_sarsa_new == q_learning_new

A terminal next state has no a-prime, so its future-value contribution is zero. The lesson also names epsilon-greedy and softmax exploration; this notebook deliberately keeps action selection supplied and visible so the comparison isolates **which next-action value each update uses**.